In [3]:
import os
import yaml
from pathlib import Path
import numpy as np
from astropy.coordinates import Angle
import astropy.units as u
from galsim.zernike import DoubleZernike

from StarSharp import StateFactory, FieldCoords

In [4]:
# Load the sensitivity matrix and normalization from MTAOS v13
mttcs_dir = Path(os.environ["TS_CONFIG_MTTCS_DIR"])
mtaos_dir = mttcs_dir / "MTAOS/v13/ofc/"
senspath = mtaos_dir / "sensitivity_matrix" / "lsst_sensitivity_dz_31_29_50.yaml"
with open(senspath, "r") as f:
    sens = np.array(yaml.safe_load(f))
normpath = mtaos_dir / "normalization_weights" / "range-fwhm.yaml"
with open(normpath, "r") as f:
    norm = np.array(yaml.safe_load(f))

In [5]:
# Sens shape is (kmax, jmax, dof_max)
# Set things we don't care about and don't want to influence the
# orthogonalization to 0.0
sens[0] = 0.0
sens[:, :4] = 0.0

In [6]:
# Create the StateFactory so we can easily convert between
# used_dof, full_dof, and orthogonalized representations of the state.
state_factory = StateFactory(sens, norm=norm, use_dof="0-9,10-16,30-34", nkeep=12)

In [7]:
# Build a random state in the vmode basis so it's guaranteed to be able to roundtrip
# through x and f bases.  Not worried about plausibility of values here.
rng = np.random.default_rng(123)
state = state_factory.from_v(rng.normal(size=state_factory.nkeep))

In [8]:
# Get the OCS WFS coordinates
rtp = Angle("0.25 rad")
wfs_coords = FieldCoords(
    x=[-1.2, -1.2, 1.2, 1.2] * u.deg,
    y=[-1.2, 1.2, -1.2, 1.2] * u.deg,
    frame="ccs",
    rtp=rtp,
)

In [9]:
# Assemble sens as list of DZs so we can evaluate it at specific coordinates.
dzs = [
    DoubleZernike(
        sens[..., idof],
        uv_outer=1.75,
        xy_inner=0.62*4.18,
        xy_outer=4.18
    )
    for idof in range(50)
]

In [10]:
# Zk true.  Add up contribution from each DOF.
zk_true = np.sum(
    [
        dzs[i].xycoef(
            wfs_coords.ocs.x.to_value(u.deg),
            wfs_coords.ocs.y.to_value(u.deg)
        ) * state.f.value[i]
        for i in range(50)
    ],
    axis=0
)

In [11]:
# Now need the sens at specific field coordinates.
vmode_dzs = [
    DoubleZernike(
        state_factory.Av[..., imode],
        uv_outer=1.75,
        xy_inner=0.62*4.18,
        xy_outer=4.18
    )
    for imode in range(state_factory.nkeep)
]
sens_1d = np.array([
    dz.xycoef(wfs_coords.ocs.x.to_value(u.deg), wfs_coords.ocs.y.to_value(u.deg))
    for dz in vmode_dzs
])

In [12]:
coefs, *_ = np.linalg.lstsq(sens_1d.reshape(state_factory.nkeep, -1).T, zk_true.ravel(), rcond=None)

In [13]:
print(coefs)
print(state.v.value)